# The factor spine: the published spine and the constructed block

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.factors.spine`

**Modules covered** `factors/spine.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The factor layer regresses each sleeve on a small set of factors. Those factors are two things kept apart: a published spine of market, size, value, profitability, investment and momentum legs, and a constructed block of four bond and credit series built from the panel's own sleeves. The entry point builds both and prints their coverage.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `factors/spine.py::constructed_block` traces to the constructed block of this effort: the sleeves' income and credit lines assembled so the block's loadings are identities
- `factors/spine.py::cross_check` traces to the named spine and the constructed block of this effort: published index legs beside a block assembled from the sleeves' own income and credit lines, kept separate so a block loading is an identity rather than a finding
- `factors/spine.py::main` traces to the named spine and the constructed block of this effort: published index legs beside a block assembled from the sleeves' own income and credit lines, kept separate so a block loading is an identity rather than a finding
- `factors/spine.py::named_set` traces to the named spine of this effort: published index legs, with the archive's own risk-free column refused as a factor because it is the series every excess return is already taken against

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`factors/spine.py`**

The named factor set: a published spine, an internally constructed block, and the two
joined into the set every other factor result is measured against.

The spine is the Fama/French Developed five-factor model plus momentum, EUR-translated,
with the Europe set carried as a cross-check. It is equity-only by construction: the
library's bond, maturity and rating portfolio files no longer exist, so it cannot describe
this mandate on its own.

The non-equity half is therefore built here, from the panel's own sleeves, and declared
**constructed** rather than presented as a vendor factor. Constructed means each series is
a difference of the panel's own returns: nothing is estimated, and nothing is assumed
about a yield curve the snapshot does not carry. With two government sleeves the block's
level and slope are that block's sum and difference and nothing further is identifiable,
so these four series are the whole term/credit block rather than four choices among many.

The input is the excess-return frame, never a gross one. A difference of excess returns is
the difference of the gross returns, because the cash rate cancels exactly; only the level
carries the cash rate, and it should, since it stands in for the government block's own
excess return.

## 3. The data contract it consumes, and the as-of rule

The spine arrives as monthly euro total returns from the archive's own files, with the archive's risk-free column refused as a factor because every sleeve's return is already taken against it. The block is arithmetic on the panel: the government level is the average of the two government sleeves, the term slope is long minus short, credit is investment grade over government, and high-yield excess is high yield over investment grade. The named set is the inner join of the two, and a missing value anywhere in it is refused rather than filled.

## 4. The worked example on small numbers, with the identity checked

The block is four lines of arithmetic, which is the point of constructing it: a load of one on the government level is the sleeve's own construction rather than a fitted estimate. The worked example builds the block from four hand-chosen sleeve returns.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import pandas as pd

from portfolio_workbench.factors import spine

sleeves = pd.DataFrame(
    {"IBGL.AS": [0.02, 0.01], "IEGE.AS": [0.01, 0.005], "IEAC.AS": [0.025, 0.012], "IHYG.L": [0.03, 0.02]}
)
block = spine.constructed_block(sleeves)

assert list(block.columns) == ["government_level", "term_slope", "credit", "high_yield_excess"]
assert block.loc[0, "government_level"] == 0.5 * (0.02 + 0.01)
assert block.loc[0, "term_slope"] == 0.02 - 0.01
assert block.loc[0, "credit"] == 0.025 - 0.02
assert block.loc[0, "high_yield_excess"] == 0.03 - 0.025
print(block.round(4))

   government_level  term_slope  credit  high_yield_excess
0            0.0150       0.010   0.005              0.005
1            0.0075       0.005   0.002              0.008


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.factors import spine

print("constructed series: " + ", ".join(spine.CONSTRUCTED))
print(f"block sleeves: {', '.join(spine.BLOCK_SLEEVES)}")
print(f"columns refused as factors: {', '.join(spine.NOT_A_FACTOR)}")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
constructed series: government_level, term_slope, credit, high_yield_excess
block sleeves: IBGL.AS, IEGE.AS, IEAC.AS, IHYG.L
columns refused as factors: RF


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.factors.spine"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[factor] snapshot 2026-09-13, sleeves in the map's order
[factor] sleeve frame 191 months × 11 sleeves, EUR total returns less the overnight rate, 2010-10..2026-08
[factor] published spine 191 months × 7 columns (Mkt-RF, SMB, HML, RMW, CMA, RF, MOM), EUR-translated; RF is the file's own risk-free and is dropped, not regressed on
[factor] the panel reaches 2010-10..2026-08, so the named set runs 2010-10..2026-07 = 190 months × 10 factors
[factor] constructed block 191 months × 4 series (government_level, term_slope, credit, high_yield_excess), built from the panel's own sleeves
    government_level     mean +0.054%/month  vol 1.67%  cumulative +7.9%
    term_slope           mean +0.116%/month  vol 3.28%  cumulative +12.6%
    credit               mean -0.018%/month  vol 2.56%  cumulative -9.2%
    high_yield_excess    mean +0.165%/month  vol 1.35%  cumulative +34.7%
[factor] named set 190 months × 10 factors, no missing value: Mkt-RF, SMB, HML, RMW, CMA, MOM, government_level, term_slop

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

Two readings decide what the rest of the factor layer can claim. The spine is six published legs on a European multi-asset panel, so a weak GRS statistic is expected and is not evidence that the factors are unpriced elsewhere. The block's loadings on the four construction sleeves are identities, so a high R-squared there measures the construction rather than the model, and only the other seven sleeves carry an alpha a reader may interpret. The block is this effort's own assembly: a reader must not read it as a vendor factor.

## 7. What this module does not establish

The spine does not establish that these six factors are the right ones, and the block does not establish a term-structure or credit model: it is four differences of sleeves, with no fitted structure behind it. Nothing here establishes that a loading is stable, since stability is a question for the rolling estimation and is answered there.